In [ ]:
# ruff: noqa: E402
# 高強度測試標配：開啟自動重載，當在外面的 src/model/ 修改程式碼時，Jupyter 會即時同步
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# 將專案根目錄加入路徑
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"[系統資訊] 成功掛載專案根目錄: {project_root}")

## Cell 2：實例化 CRNN+CTC 模型配置

In [ ]:
import torch
import torch.optim as optim
from src.data import build_default_vocabs, get_encoder_spec, collate_fn
from src.model.config import ModelConfig, LossWeights
from src.model.encoders import build_encoder
from src.model.model import OMRModel
from src.model.losses import compute_loss
from src.postproc import evaluate_batch

# 1. 建立 CTC 壓力測試配置
model_cfg = ModelConfig(
    use_ctc=True,
    decoder_layers=2,  # 2 層 BiLSTM
    rnn_hidden_dim=256,
    rnn_bidirectional=True,
    dropout=0.1,
)
model_cfg.loss_weights = LossWeights(type=1.0, pitch=1.0, rhythm=1.0, attribute=1.0)

# 2. 建立原生詞彙表與 ResNet 規格
vocabs = build_default_vocabs()
resnet_spec = get_encoder_spec("resnet")

# 3. 建構完整模型
encoder = build_encoder(resnet_spec, d_model=model_cfg.d_model)
model = OMRModel(encoder=encoder, vocabs=vocabs, cfg=model_cfg)

print("[結構驗證] CRNN+CTC 模型建構成功！")
print(
    f" -> 影像維度投影層輸入規格: {model.encoder.proj.in_features} (通道數*卷積後最終高度)"
)

## Cell 3：建構極端變長 Batch（極限壓力測試）

In [ ]:
print("[壓力測試] 開始人工建構極端變長與標籤非同步的 Batch 資料...")

# 高強度核心：模擬 DataLoader 輸出，但故意給予極限不對稱的資料
# 樣本 0：極窄影像 (240px)，標籤極短 (長度 3)
item_0 = {
    "pixel_values": torch.randn(1, 128, 240),
    "type_ids": torch.tensor([5, 6, 2]),
    "pitch_ids": torch.tensor([4, 4, 2]),
    "rhythm_ids": torch.tensor([4, 4, 2]),
    "attribute_ids": torch.tensor([5, 12, 2]),
    "label_length": 3,
}

# 樣本 1：極寬影像 (720px)，標籤很長 (長度 6)，且刻意製造多頭長度不一致（後半段用 PAD 填充）
# 模擬早期的不穩定狀態，或極端數據
item_1 = {
    "pixel_values": torch.randn(1, 128, 720),
    "type_ids": torch.tensor([6, 7, 8, 9, 2, 0]),
    "pitch_ids": torch.tensor([15, 15, 15, 2, 0, 0]),  # 音高流提早結束
    "rhythm_ids": torch.tensor([8, 8, 8, 8, 8, 2]),  # 節奏流很長
    "attribute_ids": torch.tensor([4, 4, 4, 4, 2, 0]),
    "label_length": 6,
}

# 調用不允許更動的 src/data/dataloader.py 裡的 collate_fn 進行極限對齊
mock_batch = collate_fn([item_0, item_1])

print("\n[形狀排查] 經過 collate_fn 對齊後的張量維度:")
print(
    f" -> batch['pixel_values'].shape : {list(mock_batch['pixel_values'].shape)} (寬度應自動補齊至 720)"
)
print(
    f" -> batch['type_ids'].shape     : {list(mock_batch['type_ids'].shape)} (長度應自動補齊至 6)"
)
print(
    f" -> batch['label_lengths']      : {mock_batch['label_lengths'].tolist()} (保留真實長度合約)"
)

# 驗證 collate_fn 的補零白色像素 (+1.0) 是否在窄影像區域正確生效
assert (
    mock_batch["pixel_values"][0, :, :, 240:].eq(1.0).all()
), "Data 層影像補零邊界溢出！"
print(" -> [成功] collate_fn 影像與標籤補齊邊界完全正確。")

## Cell 4：微型訓練收斂優化測試（驗證梯度與反向傳播）

In [ ]:
print("[優化測試] 開始執行 5 步微型訓練過擬合迴圈，驗證神經網路連通性與收斂性...")

# 實例化 AdamW 優化器，將 ResNet 與 BiLSTM 權重全部納入
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

model.train()
labels = {
    "type": mock_batch["type_ids"],
    "pitch": mock_batch["pitch_ids"],
    "rhythm": mock_batch["rhythm_ids"],
    "attribute": mock_batch["attribute_ids"],
}

loss_history = []

for step in range(5):
    optimizer.zero_grad()

    # 執行 Forward
    out = model(mock_batch["pixel_values"])

    # 執行步驟四修改後的 compute_loss (完美接軌 batch["label_lengths"])
    total_loss, per_head_loss = compute_loss(
        logits=out["logits"],
        labels=labels,
        cfg=model_cfg,
        target_lengths=mock_batch["label_lengths"],
    )

    # 反向傳播與優化
    total_loss.backward()

    # 高強度檢查：在 step 之前，動態檢查 ResNet 投影層與 LSTM 的梯度是否存在
    if step == 0:
        assert model.encoder.proj.weight.grad is not None, "ResNet 投影層梯度斷裂！"
        assert model.decoder.rnn.weight_hh_l0.grad is not None, "LSTM 解碼層梯度斷裂！"
        print(" -> [成功] 經檢查，ResNet 與 LSTM 梯度運算圖完全連通。")

    optimizer.step()
    loss_history.append(total_loss.item())
    print(f"  - Step {step+1}/5 | Total CTC Loss: {total_loss.item():.4f}")

print("\n[數學驗證] 微型收斂測試完成！")
print(
    f" -> 初始 Loss: {loss_history[0]:.4f} ---> 5 步更新後 Loss: {loss_history[-1]:.4f}"
)
if loss_history[-1] < loss_history[0]:
    print(" -> [優化成功] Loss 順利下降，模型具備動態時序學習能力！")
else:
    print(" -> [警告] Loss 未能下降，請檢查學習率或初始化狀態。")

## Cell 5：非自迴歸解碼與後處理指標合約驗證

In [ ]:
print("[解碼測試] 切換至 Eval 模式，執行 model.generate() 壓力解碼...")
model.eval()

with torch.no_grad():
    gen_out = model.generate(mock_batch["pixel_values"], max_length=32)

print(f" -> 解碼張量型態 : {type(gen_out)}")
print(
    f" -> Type 解碼形狀: {list(gen_out.type_ids.shape)} (應自動處理 4 頭非同步最大長度)"
)
print(f" -> 封裝長度向量 : {gen_out.lengths.tolist()}")

# 將非自迴歸輸出與 Ground Truth 打包成 list[IdSeqs] 結構
B = mock_batch["pixel_values"].shape[0]
gt_batch = [
    (
        mock_batch["type_ids"][i],
        mock_batch["pitch_ids"][i],
        mock_batch["rhythm_ids"][i],
        mock_batch["attribute_ids"][i],
    )
    for i in range(B)
]
pred_batch = [
    (
        gen_out.type_ids[i],
        gen_out.pitch_ids[i],
        gen_out.rhythm_ids[i],
        gen_out.attribute_ids[i],
    )
    for i in range(B)
]

# 呼叫 Member C 寫的、完全沒改過的原生後處理指標模組
print("\n[合約驗證] 將非自迴歸矩陣丟入 src.postproc.evaluate_batch...")
metrics = evaluate_batch(gt_batch=gt_batch, pred_batch=pred_batch, vocab=vocabs)

print("--------------------------------------------------")
print(
    "[高強度整合測試通過!] 模型在動態變長影像與非同步標籤下，完全未觸發任何 Runtime 崩潰！"
)
print(f" -> 最終綜合序列錯誤率 (SER) : {metrics.ser:.4f}")